# Agentic AI to help us to make final judgement
In this notebook, we will one of the startegy (similar to `2_backtesting_py_exp`) and after backtesting; we will use it for read trading. Our final decision will always be made by LLM agent.

Install ta-lib on Mac with brew first `brew install ta-lib` and then `pip install TA-Lib`

In [6]:
import pandas as pd
import pandas_ta as ta
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA, GOOG
import yfinance as yf
import talib 
from utils import get_stock_data
import time
import warnings

## Typical TA
Using from `ta-lib` library.

In [ ]:
class RsiBollingerStrategy(Strategy):
    rsi_period = 14
    rsi_buy = 70
    rsi_sell = 30
    bb_period = 20
    bb_std = 2
    stop_loss_pct = 0.02
    take_profit_pct = 0.05

    def init(self):
        df = self.data.df

        # Indicators from pandas_ta
        self.rsi = self.I(lambda: df.ta.rsi(length=self.rsi_period))
        bb = df.ta.bbands(length=self.bb_period, std=self.bb_std)

        # Use lower & upper bands from BB
        self.bb_lower = self.I(lambda: bb['BBL_20_2.0'])
        self.bb_upper = self.I(lambda: bb['BBU_20_2.0'])

    def next(self):
        price = self.data.Close[-1]

        # Buy condition: RSI < 30 and price below lower BB
        if self.rsi[-1] < self.rsi_buy and price < self.bb_lower[-1]:
            sl = price * (1 - self.stop_loss_pct)
            tp = price * (1 + self.take_profit_pct)
            self.buy(sl=sl, tp=tp)

        # Sell condition: RSI > 70 or price > upper BB
        elif self.position:
            if self.rsi[-1] > self.rsi_sell or price > self.bb_upper[-1]:
                self.position.close()

# Fetch data
df = get_stock_data("MSFT", interval="15m", start="2024-02-01", end="2025-03-31")

# Run backtest
bt = Backtest(df, RsiBollingerStrategy, cash=10_000, commission=0.002)
stats = bt.run()

print(stats)
# bt.plot(filename="RsiBollingerStrategy.html")

⚠️ Interval '15m' only supports ~60 days. Switching to last 60 days.


Backtest.run:   0%|          | 0/1540 [00:00<?, ?it/s]/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/backtesting.py:955: UserWarning: time=54: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/backtesting.py:955: UserWarning: time=109: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/backtesting.py:955: UserWarning: time=148: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/backtesting.py:955: UserWarning: time=149: Broker canceled the relative-sized order due to insufficient margin.
  warnings.warn(
/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/backtesting.py:955: UserWarning: time=158

Start                     2025-01-16 14:30...
End                       2025-04-11 19:45...
Duration                     85 days 05:15:00
Exposure Time [%]                    11.47436
Equity Final [$]                    8499.3466
Equity Peak [$]                       10000.0
Commissions [$]                    1652.99996
Return [%]                          -15.00653
Buy & Hold Return [%]                -9.14392
Return (Ann.) [%]                   -49.48513
Volatility (Ann.) [%]                 5.45414
CAGR [%]                            -38.17185
Sharpe Ratio                         -9.07295
Sortino Ratio                        -4.74968
Calmar Ratio                         -2.63843
Alpha [%]                           -14.38824
Beta                                  0.06762
Max. Drawdown [%]                   -18.75554
Avg. Drawdown [%]                   -18.75554
Max. Drawdown Duration       84 days 23:00:00
Avg. Drawdown Duration       84 days 23:00:00
# Trades                          

It is highly possible that one strategy works on single stock and can fail on other stock. Therefore, quickly do the back test on some more stocks.

In [ ]:
def run_multi_stock_backtest(tickers,  period="60d"):
    results = []

    for ticker in tickers:
        time.sleep(2.3)
        print(f"Running backtest for {ticker}...")
        try:
            df = get_stock_data(ticker, interval="15m", period="60d")

            if df.empty:
                continue

            bt = Backtest(df, RsiBollingerStrategy, cash=10_000, commission=0.002)     
            stats = bt.run()

            results.append({
                "Ticker": ticker,
                "Return [%]": stats['Return [%]'],
                "Win Rate [%]": stats['Win Rate [%]'],
                "Sharpe Ratio": stats.get('Sharpe Ratio', None),
                "Trades": stats['# Trades'],
                "Avg Trade Duration": stats['Avg. Trade Duration'],
                "Exposure Time [%]": stats['Exposure Time [%]'],
            })

        except Exception as e:
            print(f"Error with {ticker}: {e}")

    df_results = pd.DataFrame(results)
    #df_results.sort_values(by="Return [%]", ascending=False, inplace=True)
    return df_results

tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NFLX", "META", "NVDA", "AMD", "INTC"]
results = run_multi_stock_backtest(tickers,  period="60d")
print(results)

## Integrating ESN model from `1a_esn_le2e.ipynb`
I further experimented with ESN in ``1a_esn_le2e.ipynb` and want to use this only for future. So backtesting:

In [116]:
from utils import StockPricePredictor

predictor = StockPricePredictor(model_path="esnmodel_MSFT_4_60m_2021-01-01_2024-12-31.pkl", 
                                scaler_path="scaler_MSFT_4_60m_2021-01-01_2024-12-31.pkl")

def if_buy_signal_with_esn(df, threshold=0.01):
    """
    Determines if a BUY signal should be generated based on the ESN model's prediction.
    
    Args:
        df (DataFrame): The dataframe containing the latest stock data.
        threshold (float): The minimum percentage increase required to trigger a BUY signal.
    
    Returns:
        int: 1 if BUY signal is triggered, otherwise 0.
    """
    predicted_close = predictor.predict(df)
    last_close = df.iloc[-1]['Close']
    
    # Check if predicted close is at least (1 + threshold)% higher than last close
    if predicted_close >= last_close * (1 + threshold):
        return 1  # BUY Signal
    
    return 0  # No action


class MLTradingStrategy(Strategy):
    feature_window = 4 # window size for features i.e., hisorical data length
    stop_loss_pct = 0.02
    take_profit_pct = 0.04

    def init(self):
        pass

    def next(self):
        N = self.feature_window  
        # Ensure enough data to generate features
        if len(self.data.Close) < N:
            return 

        # Extract last `N` rows into a DataFrame
        df = pd.DataFrame({
            "Open": self.data.Open[-N:],
            "High": self.data.High[-N:],
            "Low": self.data.Low[-N:],
            "Close": self.data.Close[-N:],
            #"Volume": self.data.Volume[-N:],
        })

        # Pass last `N` rows to ML model
        signal = if_buy_signal_with_esn(df)  

        # ML suggests BUY signal
        if signal == 1:  
            price = self.data.Close[-1]
            sl = price * (1 - self.stop_loss_pct)  
            tp = price * (1 + self.take_profit_pct) 
            self.buy(sl=sl, tp=tp)  

df = get_stock_data("MSFT", interval="60m", start="2024-02-01", end="2025-03-31")

# Run Backtest
bt = Backtest(df, MLTradingStrategy, cash=10000, commission=0.002)
stats = bt.run()

# Print Results
print(stats)
bt.plot(filename="MLTradingStrategy.html")

⚠️ Interval '60m' only supports ~60 days. Switching to last 60 days.


/var/folders/vd/m8tzs4_92z538swpylf0xtvc0000gn/T/ipykernel_49061/2874217586.py:68: UserWarning: Superimposed OHLC plot matches the original plot. Skipping.
  bt.plot(filename="MLTradingStrategy.html")


Start                     2025-01-16 14:30...
End                       2025-04-11 19:30...
Duration                     85 days 05:00:00
Exposure Time [%]                     9.04762
Equity Final [$]                  10378.12578
Equity Peak [$]                   10415.36052
Commissions [$]                     197.93655
Return [%]                            3.78126
Buy & Hold Return [%]                -8.95139
Return (Ann.) [%]                    16.86905
Volatility (Ann.) [%]                 15.7277
CAGR [%]                             11.60176
Sharpe Ratio                          1.07257
Sortino Ratio                         1.85288
Calmar Ratio                          2.27643
Alpha [%]                             4.92035
Beta                                  0.12725
Max. Drawdown [%]                     -7.4103
Avg. Drawdown [%]                    -2.10044
Max. Drawdown Duration        8 days 04:00:00
Avg. Drawdown Duration        2 days 08:00:00
# Trades                          

GridPlot(id='p2959', ...)

## Using LLM agent to decide for future (live trading)
Here, I use creaAI and OpenAI's model for the LLM. My OpenAI key is store in `.env` file.

In [ ]:
from dotenv import load_dotenv
import os
from crewai import Crew, Agent, Task
from langchain_openai import ChatOpenAI
import json
from langchain_core.caches import BaseCache
from langchain_core.caches import InMemoryCache

load_dotenv()
try:
    os.environ["OPENAI_API_KEY"]
except KeyError:
    print("Please provide OPEN_API_KEY as env variable!")

In [ ]:
# 1. Define the LLM (with cache initialization)
BaseCache.global_cache = InMemoryCache()
ChatOpenAI.model_rebuild()

llm = ChatOpenAI(model="gpt-4o") # Using ChatGPT-4o

# 2. Define the Agent
trading_analyst = Agent(
    role='Trading Analyst',
    goal='Evaluate stock market data and provide actionable trading recommendations with risk management parameters.',
    backstory='A seasoned financial analyst with expertise in both quantitative and qualitative analysis of stock markets.',
    llm=llm,
    allow_delegation=False # This agent will perform the final evaluation
)

# 3. Define the Task
def create_trading_evaluation_task(predicted_price, backtesting_results, market_sentiment):
    task_description = f"""
    You are a Trading Analyst tasked with evaluating the potential trading opportunity for a stock.
    You will be provided with the following information:

    **1. ML Model Prediction:** The predicted closing price of the stock in the next 60 minutes is: {predicted_price}

    **2. Backtesting Results:** Here's how a similar trading strategy has performed over the last 2-3 months:
    ```
    {backtesting_results}
    ```

    **3. Market Sentiment:** Recent information and news indicating the current market sentiment:
    ```
    {market_sentiment}
    ```

    Based on this information, you need to provide a trading recommendation in JSON format with the following keys:
    "action" (either "BUY" or "SELL"),
    "stop_loss" (the price at which to exit the trade to limit losses),
    "take_profit" (the price at which to exit the trade to secure profits),
    "reasoning" (a concise explanation of your recommendation).

    Your analysis should consider the predicted price movement, the historical performance of the strategy, and the prevailing market sentiment.
    Ensure the stop loss and take profit levels are reasonable and justified based on your analysis.

    Return your analysis as a valid JSON object.
    """
    return Task(
        description=task_description,
        agent=trading_analyst,
        expected_output='''{"action": "BUY" or "SELL", "stop_loss": number, "take_profit": number, "reasoning": "string"}'''
    )

# 4. Create the Crew
def create_trading_crew(predicted_price, backtesting_results, market_sentiment):
    task = create_trading_evaluation_task(predicted_price, backtesting_results, market_sentiment)
    crew = Crew(
        agents=[trading_analyst],
        tasks=[task],
        verbose=True # Changed from 2 to True
    )
    return crew

# 5. Example Usage
if __name__ == "__main__":
    # Example input data 
    predicted_closing_price = 155.75
    backtesting_data = """
    Total Trades: 50
    Winning Trades: 35 (70%)
    Losing Trades: 15 (30%)
    Average Profit per Trade: $0.80
    Average Loss per Trade: -$0.30
    Maximum Drawdown: $2.50
    """
    sentiment_info = """
    Recent news indicates positive market sentiment due to better-than-expected earnings reports in the tech sector.
    However, there are concerns about rising inflation figures which could lead to potential interest rate hikes.
    """
    # Create and run the crew
    trading_crew = create_trading_crew(predicted_closing_price, backtesting_data, sentiment_info)
    result = trading_crew.kickoff()


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bf787c7a-4f18-4cf2-9df1-39f951793a1c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: ddce7c00-e588-43a4-bd84-969f561cf8f4
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: ddce7c00-e588-43a4-bd84-969f561cf8f4
       Status: Executing Task...
    └── 🤖 Agent: Trading Analyst
            Status: In Progress

# Agent: Trading Analyst
## Task: 
    You are a Trading Analyst tasked with evaluating the potential trading opportunity for a stock.
    You will be provided with the following information:

    **1. ML Model Prediction:** The predicted closing price of the stock in the next 60 minutes is: 155.75

    **2. Backtesting Results:** Here's how a similar trading strategy has performed over the last 2-3 months:
    ```
    
    Total Trades: 50
    Winning Trades: 35 (70%)
    Losing Trades: 15 (30%)
    Average Profit per Trade: $0.80
    Average Loss per Trade: -$0.30
    Maximum Drawdown: $2.50
    
    ```

    **3. Market Sentiment:** Recent information and news indicating the current market sentiment:
    ```
    
    Recent news indicates positive market sentiment due to better-than-expected earnings reports in the tech sector.
    However, there are concerns about rising inflation figures which could lead to potential interest rate hikes.
    
    ```

    Based on this information

🚀 Crew: crew
└── 📋 Task: ddce7c00-e588-43a4-bd84-969f561cf8f4
       Status: Executing Task...
    └── 🤖 Agent: Trading Analyst
            Status: ✅ Completed

🚀 Crew: crew
└── 📋 Task: ddce7c00-e588-43a4-bd84-969f561cf8f4
       Assigned to: Trading Analyst
       Status: ✅ Completed
    └── 🤖 Agent: Trading Analyst
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: ddce7c00-e588-43a4-bd84-969f561cf8f4                                                                     │
│  Agent: Trading Analyst                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bf787c7a-4f18-4cf2-9df1-39f951793a1c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
print(result)

```json
{
    "action": "BUY",
    "stop_loss": 153.25,
    "take_profit": 157.50,
    "reasoning": "The ML model predicts an increase in stock price to 155.75 within the next 60 minutes. The backtested strategy shows a solid winning ratio of 70%, with a historical average gain outweighing the losses significantly, which provides a good risk-reward ratio. The positive market sentiment driven by strong tech sector earnings further supports a bullish outlook despite concerns regarding inflation and interest rates. Setting the stop loss at 153.25 limits the downside to $2.50, aligning with the maximum historical drawdown, while setting a take-profit target at 157.50 ensures capturing gains above the predicted price increase."
}
```


END OF NOTEBOOK